# 🚀 Project Completion Predictor — ML Pipeline

End-to-end notebook: data loading → cleaning → feature engineering → EDA → model training → evaluation → saving.

In [ ]:
import pandas as pd
import numpy as np
import json
import joblib
import os
import warnings
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
from datetime import datetime
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
print('Libraries loaded')

## 1. Data Loading

In [ ]:
df = pd.read_csv('users_large_dataset.csv')
print(f'Rows: {len(df)} | Columns: {list(df.columns)}')
df.head(2)

## 2. Data Cleaning

In [ ]:
# Date parsing
for col in ['createdAt','updatedAt','dob']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Drop columns not needed for ML
safe_df = df.drop(columns=['password','profileUrl'], errors='ignore')

print('Missing values:')
print(safe_df.isnull().sum())
print(f'\nVerified users : {df.isVerified.sum()}')
print(f'Deleted users  : {df.isDeleted.sum()}')
safe_df.dtypes

## 3. Parse & Normalise Projects Column

In [ ]:
NOW = pd.Timestamp('2026-11-15')

rows = []
for _, user in df.iterrows():
    try:
        projs = json.loads(user['projects'])
    except:
        continue
    for p in projs:
        tasks = p.get('tasks', [])
        total_tasks    = len(tasks)
        completed      = sum(1 for t in tasks if t.get('progress', 0) >= 100)
        avg_prog       = np.mean([t.get('progress', 0) for t in tasks]) if tasks else 0
        pc = pd.to_datetime(p.get('createdAt'), errors='coerce')
        pu = pd.to_datetime(p.get('updatedAt'), errors='coerce')
        end_dates = [pd.to_datetime(t.get('endDate'), errors='coerce')
                     for t in tasks if pd.notnull(pd.to_datetime(t.get('endDate'), errors='coerce'))]
        planned_end = max(end_dates) if end_dates else None
        rows.append({
            'user_id':          user['id'],
            'username':         user['username'],
            'project_id':       p['id'],
            'project_name':     p['projectName'],
            'is_deleted':       p.get('isDeleted', False),
            'proj_created':     pc,
            'proj_updated':     pu,
            'planned_end':      planned_end,
            'total_tasks':      total_tasks,
            'completed_tasks':  completed,
            'avg_progress':     avg_prog,
            'is_verified':      user['isVerified'],
        })

proj_df = pd.DataFrame(rows)
print(f'Total projects parsed: {len(proj_df)}')
proj_df.head(3)

## 4. Feature Engineering

In [ ]:
proj_df['project_age_days']      = (NOW - proj_df['proj_created']).dt.days.fillna(0)
proj_df['days_since_updated']    = (NOW - proj_df['proj_updated']).dt.days.fillna(0)
proj_df['completion_pct']        = proj_df['avg_progress']
proj_df['task_completion_ratio'] = proj_df.apply(
    lambda r: r['completed_tasks']/r['total_tasks'] if r['total_tasks'] > 0 else 0, axis=1)
proj_df['remaining_days']        = proj_df['planned_end'].apply(
    lambda d: max(0, (d - NOW).days) if pd.notnull(d) else 0)

print('New features added:')
proj_df[['project_age_days','days_since_updated','completion_pct',
          'task_completion_ratio','remaining_days']].describe()

## 5. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Project Dataset — EDA', fontsize=16, fontweight='bold')

# 5.1 Verified vs Unverified
vc = df['isVerified'].value_counts()
axes[0,0].pie(vc, labels=['Verified','Unverified'], autopct='%1.1f%%',
              colors=['#4CAF50','#FF7043'], startangle=90)
axes[0,0].set_title('User Verification Status')

# 5.2 Projects per user distribution
proj_counts = proj_df.groupby('username')['project_id'].count()
axes[0,1].hist(proj_counts, bins=20, color='#2196F3', edgecolor='white')
axes[0,1].set_title('Projects per User')
axes[0,1].set_xlabel('# Projects'); axes[0,1].set_ylabel('Users')

# 5.3 Average progress distribution
axes[0,2].hist(proj_df['completion_pct'], bins=20, color='#9C27B0', edgecolor='white')
axes[0,2].set_title('Avg Task Progress Distribution')
axes[0,2].set_xlabel('Progress %')

# 5.4 Active vs Deleted projects
del_counts = proj_df['is_deleted'].value_counts()
axes[1,0].bar(['Active','Deleted'], del_counts.values,
              color=['#00BCD4','#F44336'])
axes[1,0].set_title('Project Status'); axes[1,0].set_ylabel('Count')

# 5.5 Remaining days distribution
axes[1,1].hist(proj_df['remaining_days'], bins=30, color='#FF9800', edgecolor='white')
axes[1,1].set_title('Remaining Days Distribution')
axes[1,1].set_xlabel('Days')

# 5.6 Task completion ratio
axes[1,2].scatter(proj_df['completion_pct'], proj_df['remaining_days'],
                  alpha=0.3, color='#607D8B', s=20)
axes[1,2].set_title('Progress vs Remaining Days')
axes[1,2].set_xlabel('Avg Progress %'); axes[1,2].set_ylabel('Remaining Days')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=120, bbox_inches='tight')
plt.close()
print('EDA plots saved')

## 6. Model Training

In [ ]:
FEATURES = ['project_age_days','days_since_updated','completion_pct',
            'task_completion_ratio','total_tasks','completed_tasks']
TARGET   = 'remaining_days'

valid = proj_df[FEATURES + [TARGET]].dropna()
X, y  = valid[FEATURES], valid[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

models = {
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    results[name] = {
        'model': m,
        'MAE':   mean_absolute_error(y_test, pred),
        'MSE':   mean_squared_error(y_test, pred),
        'RMSE':  np.sqrt(mean_squared_error(y_test, pred)),
        'R2':    r2_score(y_test, pred),
    }
    print(f'{name:<22} MAE={results[name]["MAE"]:.3f}  RMSE={results[name]["RMSE"]:.3f}  R2={results[name]["R2"]:.4f}')

## 7. Model Evaluation

In [ ]:
best_name  = min(results, key=lambda n: results[n]['MAE'])
best       = results[best_name]
best_model = best['model']
pred_best  = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Model Evaluation — {best_name}', fontsize=14, fontweight='bold')

# Actual vs Predicted
axes[0].scatter(y_test, pred_best, alpha=0.5, color='#1565C0', s=25)
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn,mx],[mn,mx],'r--', linewidth=2)
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].set_title('Actual vs Predicted')

# Metrics comparison bar
metric_names = ['MAE','RMSE','R2']
colors = ['#E53935','#FB8C00','#43A047']
for ax_i, (mn2, col) in enumerate(zip(metric_names, colors)):
    vals = [results[n][mn2] for n in results]
    names = list(results.keys())
    axes[1].bar([f'{n}\n({mn2})' for n in names], vals, color=col, alpha=0.8, label=mn2)

axes[1].set_title('Model Metrics Comparison')
axes[1].set_ylabel('Score')

plt.tight_layout()
plt.savefig('model_eval.png', dpi=120, bbox_inches='tight')
plt.close()
print(f'Best: {best_name} | MAE={best["MAE"]:.3f} | R2={best["R2"]:.4f}')

## 8. Feature Importance

In [ ]:
fi = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 4))
fi.plot(kind='barh', color='#5C6BC0', ax=ax)
ax.set_title(f'Feature Importance — {best_name}')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.close()
print('Feature importance plotted')

## 9. Save Model

In [ ]:
joblib.dump({
    'model':    best_model,
    'features': FEATURES,
    'proj_df':  proj_df,
    'users_df': df,
}, 'model.pkl')
print('model.pkl saved')
print(f'Best model : {best_name}')
print(f'MAE        : {best["MAE"]:.3f} days')
print(f'RMSE       : {best["RMSE"]:.3f} days')
print(f'R2         : {best["R2"]:.4f}')